# Model Descriptions for Cajal

A minimal public-API example: **fitted model -> canonical description -> semantic JSON -> application**.

Choose your **Python (mnplib local)** kernel and run all cells. The first cell shows the Python executable and imported package location. No server, network access, or private serializer API is required.


In [ ]:
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import JSON, display
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier

import mnplib
from mnplib import ModelDescription, describe_model
from mnplib.models.language import execute

print("Python:", sys.executable)
print("mnplib:", mnplib.__file__)


## 1. Prepare Two Small Models

Model fitting belongs to the backend. These examples use named columns and intentionally train on columns **[2, 0]**, in that order. The exported expressions must still refer to the original columns as `x2` and `x0`.


In [ ]:
X = pd.DataFrame({
    "length": [0., 0., 1., 1., 2., 2., 3., 3.],
    "unused": [3., 0., 4., 7., 2., 5., 6., 1.],
    "width":  [0., 1., 0., 1., 0., 1., 0., 1.],
})
selected = [2, 0]
X_model = X.iloc[:, selected]

y_regression = 1 + 2 * X["length"] - 0.5 * X["width"]
y_classification = np.where(X["length"] < 2, "short", "long")

models = {
    "linear": LinearRegression().fit(X_model, y_regression),
    "tree": DecisionTreeClassifier(max_depth=1, random_state=42).fit(
        X_model, y_classification
    ),
}


## 2. Obtain Canonical Descriptions

`describe_model()` is the common public entry point. Feature names are inferred from the fitted DataFrame columns; `feature_names=[...]` can also be provided explicitly.

`canonical` is the computational text used for surfeit. Human-readable feature names belong to the semantic JSON and do not change this text.


In [ ]:
descriptions = {
    name: describe_model(model, feature_indices=selected)
    for name, model in models.items()
}

for name, description in descriptions.items():
    print(f"{name}: {description.canonical}")


## 3. Export the Application Payload

`to_dict()` returns ordinary JSON-compatible values. This is the boundary between mnplib and a tool such as Cajal: the application receives the payload, not fitted estimators or internal Python AST classes.

The JSON encode/decode below simulates sending the descriptions to another application. Expand the output to inspect arithmetic nodes, tree branches, class labels, feature names, and metadata.


In [ ]:
wire_json = json.dumps(
    {name: description.to_dict() for name, description in descriptions.items()},
    allow_nan=False,
)
received = json.loads(wire_json)

display(JSON(received, expanded=True))


## 4. Inspect the Payload Without Estimators

A consumer checks `schema` and `schema_version`, then dispatches on semantic node names in `ast["type"]`, such as `conditional`, `add`, or `feature`. It does not parse `canonical`.

- `ast`: prediction structure, parameters, feature references, and labels.
- `feature_indices` / `feature_names`: original-coordinate input mapping.
- `metadata`: computational facts such as tree depth or features used.
- `canonical`: optional text display; it is not the presentation schema.

Cajal owns layout, formulas, diagrams, and styling. Predictions are not embedded in this payload.


In [ ]:
for payload in received.values():
    assert payload["schema"] == "mnplib-model"
    assert payload["schema_version"] == 1
    assert payload["feature_indices"] == selected
    assert payload["feature_names"] == ["width", "length"]
    assert "predictions" not in payload

display(pd.DataFrame([
    {
        "model": name,
        "model_type": payload["model_type"],
        "root_node": payload["ast"]["type"],
        "input_columns": payload["feature_indices"],
        "features_used": payload["metadata"]["features_used"],
    }
    for name, payload in received.items()
]))


## 5. Optional Backend Verification

A Python backend can reload semantic JSON with `ModelDescription.from_dict()` and execute it with mnplib's restricted interpreter. A presentation-only client does not need this step.

Execution receives the **full original matrix `X`**, because the expressions use original column indices. Parameters are quantized to three significant digits. These deliberately simple examples retain their predictions; arbitrary models can differ from the fitted estimator, especially near decision boundaries.


In [ ]:
restored = {
    name: ModelDescription.from_dict(payload)
    for name, payload in received.items()
}
predictions = {
    name: execute(description.ast, X)
    for name, description in restored.items()
}

for name, description in restored.items():
    assert description.to_dict() == received[name]
    np.testing.assert_array_equal(
        predictions[name], execute(description.canonical, X)
    )

np.testing.assert_allclose(
    predictions["linear"], models["linear"].predict(X_model), rtol=1e-12, atol=1e-12
)
np.testing.assert_array_equal(
    predictions["tree"], models["tree"].predict(X_model)
)

display(pd.DataFrame(predictions))
print("JSON round trip, canonical execution, and prediction checks passed.")


## Using an AutoML Result

For an already fitted `NescienceClassifier`, `NescienceRegressor`, or supported `TimeSeries` search, the selected model uses the same public call:

```python
payload = describe_model(fitted_search).to_dict()
```

Do not pass `feature_indices` or `feature_names` here: the fitted search already carries its input mapping.

Schema version 1 supports the registered sklearn model families and finite-window forecasting. ARIMA and structural state-space descriptions are not supported. See [the model-language guide](SerializerGrammarExamples.md) for the complete contract.
